In [4]:
%pylab inline
%config InlineBackend.figure_format = 'retina'
from ipywidgets import interact
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# Set random seed for reproducibility
tf.random.set_seed(1)
np.random.seed(1)

from Training_Data.Particle_Tracking_Training_Data import Particle_Tracking_Training_Data
from tensorflow.keras.utils import register_keras_serializable
from tensorflow.keras import layers, models
from models.model import particle_tracking_model, compile_particle_model, load_particle_model

%pylab is deprecated, use %matplotlib inline and import the required libraries.
Populating the interactive namespace from numpy and matplotlib


# Procedurally generated training data
The code below demonstrates how to generate training videos and labels. The function also returns the ground truth particle tracks, which might also be useful depending on your goals.

Note that the training generator is a Tensorflow Module and can be easily incorperated into a Tensorflow neural network. Alternatively, you could simply save a large set of data and use another machine learning framework.

Note that the image dimension is fixed at 256x256. The labels are downsampled to 128x128 in the image dimensions. There are two classes (a particle is detected or not detected) per label so the label shape is 128x128x2. Hence, the neural network output should be 128x128x2.

In [5]:
Nt = 50 ## number of frames for each video
kappa = 0.1 ## standard deviation of background noise added to image
a = 3. ## scale factor for the size of particle spots (not true size of particles)
IbackLevel = 0.15 ## relative intensity of randomly generated background pattern; in (0, 1)
Nparticles = 10 ## the number of particles (more => slower)
sigma_motion = 2.3 ## the standard deviation for particle brownian motion; should be in (0, 10)

## you might consider randomizing some of these parameters when training a neural net

pt = Particle_Tracking_Training_Data(Nt) ## create object instance
## you can 'call' the object as many times as you want
## in this example, we only generate one training example
vid, labels, tracks = pt(kappa, a, IbackLevel, Nparticles, sigma_motion) 

## Visualizing training videos and labels

In [6]:
@interact(t=(0, Nt-1, 1))
def plotfn(t=0, show_tracks=True):
    fig = figure(1, [14, 7])
    fig.add_subplot(121)
    imshow(vid[t], origin='lower')
    if show_tracks:
        plot(tracks[t, :, 0], tracks[t, :, 1], 'rx')
    xlim(-10, 265)
    ylim(-10, 265)
    
    fig.add_subplot(122)
    imshow(vid[t], origin='lower')
    # labels are 128x128; align overlay to the 256x256 image canvas
    imshow(labels[t, ..., 1], origin='lower', cmap='spring', alpha=0.5, interpolation='nearest', extent=(0, 256, 0, 256))

interactive(children=(IntSlider(value=0, description='t', max=49), Checkbox(value=True, description='show_trac…

# Design and train a convolutional neural network using the training data generator

## Train the new model on procedurally generated data
This section builds a dataset by generating multiple short videos with `Particle_Tracking_Training_Data`, normalizes frames, splits into train/validation sets, and trains with CE+Dice loss and useful metrics (Dice/IoU/PR-AUC).

In [7]:
# Build and compile the model from the new module (CE + Dice by default)
model = particle_tracking_model()
model = compile_particle_model(model, loss_name='ce_dice', dice_weight=0.5)

In [8]:
# Build a small on-the-fly dataset and train
num_videos = 8          # increase for more data
frames_per_video = Nt   # keep from above

vids = []
lbls = []
for i in range(num_videos):
    v, l, _ = pt(kappa, a, IbackLevel, Nparticles, sigma_motion)
    # Normalize to [0,1] float32 and add channel dim
    v = tf.cast(v, tf.float32)
    vmin = tf.reduce_min(v)
    vmax = tf.reduce_max(v)
    v = (v - vmin) / tf.maximum(vmax - vmin, 1e-6)
    vids.append(v[..., None])
    lbls.append(tf.cast(l, tf.float32))  # ensure float32 labels

X = tf.concat(vids, axis=0).numpy()   # (N_total_frames, 256, 256, 1)
Y = tf.concat(lbls, axis=0).numpy()   # (N_total_frames, 128, 128, 2)

# Simple train/val split
val_ratio = 0.2
N = X.shape[0]
idx = np.arange(N)
np.random.shuffle(idx)
val_n = int(N * val_ratio)
val_idx, train_idx = idx[:val_n], idx[val_n:]
X_train, Y_train = X[train_idx], Y[train_idx]
X_val, Y_val = X[val_idx], Y[val_idx]

# Callbacks
callbacks = [
    tf.keras.callbacks.ModelCheckpoint('model_new.keras', save_best_only=True, monitor='val_loss'),
    tf.keras.callbacks.ReduceLROnPlateau(patience=3, factor=0.5, min_lr=1e-5),
    tf.keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True)
]

# Train
history = model.fit(
    X_train, Y_train,
    validation_data=(X_val, Y_val),
    epochs=20,
    batch_size=8,
    callbacks=callbacks,
    verbose=1,
)

Epoch 1/20
 3/40 ━━━━━━━━━━━━━━━━━━━━ 14s 381ms/step - dice_fg: 0.0103 - iou_fg: 0.0051 - loss: 1.5522 - pr_auc_fg: 0.0430

KeyboardInterrupt: 

In [ ]:
# Save model (if using a custom loss, provide custom_objects when loading)
# model.save('9_10.keras')

In [9]:
# Load a saved model built with the new module (handles custom objects)
model = load_particle_model('9_10.keras')

In [10]:
# Test samples
test_vid, test_labels, test_tracks = pt(kappa, a, IbackLevel, Nparticles, sigma_motion)

# Visualizing testing videos and labels
@interact(t=(0, Nt-1, 1))
def plotfn(t=0, show_tracks=True):
    fig = figure(1, [14, 7])
    fig.add_subplot(121)
    imshow(test_vid[t], origin='lower')
    if show_tracks:
        plot(test_tracks[t, :, 0], test_tracks[t, :, 1], 'rx')
    xlim(-10, 265)
    ylim(-10, 265)
    
    fig.add_subplot(122)
    imshow(test_vid[t], origin='lower')
    # Align 128x128 labels to 256x256 canvas
    imshow(test_labels[t, ..., 1], origin='lower', cmap='spring', alpha=0.5, interpolation='nearest', extent=(0, 256, 0, 256))

interactive(children=(IntSlider(value=0, description='t', max=49), Checkbox(value=True, description='show_trac…

In [18]:
# Use our model (normalize and ensure channel dimension)
# Normalize test video to [0,1] float32, same as training
_test_vid_f = tf.cast(test_vid, tf.float32)
_vmin = tf.reduce_min(_test_vid_f)
_vmax = tf.reduce_max(_test_vid_f)
test_vid_norm = (_test_vid_f - _vmin) / tf.maximum(_vmax - _vmin, 1e-6)

# Ensure channel dimension (Nt, 256, 256, 1)
if test_vid_norm.ndim == 3:
    test_vid_in = test_vid_norm[..., None]
else:
    test_vid_in = test_vid_norm

# Predict
predictions = model.predict(test_vid_in, verbose=0)

# Convert predictions to binary mask (Nt, 128, 128)
pred_mask = predictions[..., 1] > 0.5


# Interactive visualization of predictions (no threshold)
@interact(t=(0, Nt-1, 1), show_tracks=True, show_mask_only=False)
def show_prediction(t=0, show_tracks=True, show_mask_only=False):
    fig = figure(1, [14, 7])

    # Left: raw frame + tracks
    fig.add_subplot(121)
    imshow(test_vid[t], origin='lower')
    if show_tracks:
        plot(test_tracks[t, :, 0], test_tracks[t, :, 1], 'rx')
    xlim(-10, 265)
    ylim(-10, 265)

    # Right: raw frame + binary mask (or mask only)
    fig.add_subplot(122)
    if show_mask_only:
        imshow(pred_mask[t], origin='lower', cmap='gray', interpolation='nearest', extent=(0, 256, 0, 256))
        title("Binary mask")
    else:
        imshow(test_vid[t], origin='lower')
        imshow(pred_mask[t], origin='lower', cmap='spring', alpha=0.5, interpolation='nearest', extent=(0, 256, 0, 256))
        title("Binary mask overlay")

interactive(children=(IntSlider(value=0, description='t', max=49), Checkbox(value=True, description='show_trac…

In [ ]:
# (Placeholder to keep cell order tidy; you can remove this cell)